In [ ]:
import os
import shutil
import random
from tqdm import tqdm
import glob

In [ ]:
# --- CONFIGURATION ---
SOURCE_DIR = "/kaggle/input/humanml3d/HumanML3D/humanml/"
TARGET_DIR = "/kaggle/working/"
SUBSET_SIZE_PER_TYPE = 2500  # Number of M-prefixed and numeric-only IDs each (Total 2*N)
SEED = 42
MIN_FRAMES, MAX_FRAMES = 60, 200

random.seed(SEED)
os.makedirs(TARGET_DIR, exist_ok=True)

print(f"Source Directory: {SOURCE_DIR}")
print(f"Target Directory: {TARGET_DIR}")
!rm -rf kaggle/working/

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import os
import numpy as np


# --- UPDATED: Check files from new_joints instead of new_joint_vecs ---
def check_file(filename):
    # MODIFIED: Read from new_joints/ instead of new_joint_vecs/
    path = "/kaggle/input/humanml3d/HumanML3D/humanml/new_joints/" + filename
    try:
        # Only read metadata - joint positions have shape (N, 22, 3)
        data = np.load(path, mmap_mode="r")
        frames = data.shape[0]
        if MIN_FRAMES <= frames <= MAX_FRAMES:
            return filename[:-4]
    except:
        return None


# MODIFIED: List files from new_joints/ directory
all_files = [
    f
    for f in os.listdir("/kaggle/input/humanml3d/HumanML3D/humanml/new_joints/")
    if f.endswith(".npy")
]

# Use all available CPU cores
with ProcessPoolExecutor() as executor:
    results = list(executor.map(check_file, all_files))

# Filter out the Nones
valid_ids = [r for r in results if r is not None]

# 1. Load all valid IDs
m_ids = [i for i in valid_ids if i.startswith("M")]
numeric_ids = [i for i in valid_ids if not i.startswith("M")]

print(f"Total IDs in dataset: {len(valid_ids)}")
print(f"  - 'M' prefixed: {len(m_ids)}")
print(f"  - Numeric-only: {len(numeric_ids)}")

# 2. Sample equal numbers
random.seed(SEED)
n = min(SUBSET_SIZE_PER_TYPE, len(m_ids), len(numeric_ids))
selected_m = random.sample(m_ids, n)
selected_numeric = random.sample(numeric_ids, n)

subset_ids = sorted(selected_m + selected_numeric)
print(f"\nSelected subset of {len(subset_ids)} IDs ({n} 'M' IDs and {n} numeric IDs).")
subset_ids[:50]

In [ ]:
# =========================================================
# CLOUD ENVIRONMENT SETUP - EMBEDDED UTILITIES
# =========================================================
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# Always create utils files (for both cloud and local)
print("Setting up utility files...")

FILES = {
    "utils/__init__.py": "# Utils module\n",
    "utils/quaternion.py": '''# Copyright (c) 2018-present, Facebook, Inc.
# All rights reserved.
#
# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.
#

import torch
import numpy as np

_EPS4 = np.finfo(float).eps * 4.0

_FLOAT_EPS = np.finfo(np.float64).eps


# PyTorch-backed implementations
def qinv(q):
    assert q.shape[-1] == 4, "q must be a tensor of shape (*, 4)"
    mask = torch.ones_like(q)
    mask[..., 1:] = -mask[..., 1:]
    return q * mask


def qinv_np(q):
    assert q.shape[-1] == 4, "q must be a tensor of shape (*, 4)"
    return qinv(torch.from_numpy(q).float()).numpy()


def qnormalize(q):
    assert q.shape[-1] == 4, "q must be a tensor of shape (*, 4)"
    return q / torch.norm(q, dim=-1, keepdim=True)


def qmul(q, r):
    """
    Multiply quaternion(s) q with quaternion(s) r.
    Expects two equally-sized tensors of shape (*, 4), where * denotes any number of dimensions.
    Returns q*r as a tensor of shape (*, 4).
    """
    assert q.shape[-1] == 4
    assert r.shape[-1] == 4

    original_shape = q.shape

    # Compute outer product
    terms = torch.bmm(r.view(-1, 4, 1), q.view(-1, 1, 4))

    w = terms[:, 0, 0] - terms[:, 1, 1] - terms[:, 2, 2] - terms[:, 3, 3]
    x = terms[:, 0, 1] + terms[:, 1, 0] - terms[:, 2, 3] + terms[:, 3, 2]
    y = terms[:, 0, 2] + terms[:, 1, 3] + terms[:, 2, 0] - terms[:, 3, 1]
    z = terms[:, 0, 3] - terms[:, 1, 2] + terms[:, 2, 1] + terms[:, 3, 0]
    return torch.stack((w, x, y, z), dim=1).view(original_shape)


def qrot(q, v):
    """
    Rotate vector(s) v about the rotation described by quaternion(s) q.
    Expects a tensor of shape (*, 4) for q and a tensor of shape (*, 3) for v,
    where * denotes any number of dimensions.
    Returns a tensor of shape (*, 3).
    """
    assert q.shape[-1] == 4
    assert v.shape[-1] == 3
    assert q.shape[:-1] == v.shape[:-1]

    original_shape = list(v.shape)
    # print(q.shape)
    q = q.contiguous().view(-1, 4)
    v = v.contiguous().view(-1, 3)

    qvec = q[:, 1:]
    uv = torch.cross(qvec, v, dim=1)
    uuv = torch.cross(qvec, uv, dim=1)
    return (v + 2 * (q[:, :1] * uv + uuv)).view(original_shape)


def qeuler(q, order, epsilon=0, deg=True):
    """
    Convert quaternion(s) q to Euler angles.
    Expects a tensor of shape (*, 4), where * denotes any number of dimensions.
    Returns a tensor of shape (*, 3).
    """
    assert q.shape[-1] == 4

    original_shape = list(q.shape)
    original_shape[-1] = 3
    q = q.view(-1, 4)

    q0 = q[:, 0]
    q1 = q[:, 1]
    q2 = q[:, 2]
    q3 = q[:, 3]

    if order == "xyz":
        x = torch.atan2(2 * (q0 * q1 - q2 * q3), 1 - 2 * (q1 * q1 + q2 * q2))
        y = torch.asin(torch.clamp(2 * (q1 * q3 + q0 * q2), -1 + epsilon, 1 - epsilon))
        z = torch.atan2(2 * (q0 * q3 - q1 * q2), 1 - 2 * (q2 * q2 + q3 * q3))
    elif order == "yzx":
        x = torch.atan2(2 * (q0 * q1 - q2 * q3), 1 - 2 * (q1 * q1 + q3 * q3))
        y = torch.atan2(2 * (q0 * q2 - q1 * q3), 1 - 2 * (q2 * q2 + q3 * q3))
        z = torch.asin(torch.clamp(2 * (q1 * q2 + q0 * q3), -1 + epsilon, 1 - epsilon))
    elif order == "zxy":
        x = torch.asin(torch.clamp(2 * (q0 * q1 + q2 * q3), -1 + epsilon, 1 - epsilon))
        y = torch.atan2(2 * (q0 * q2 - q1 * q3), 1 - 2 * (q1 * q1 + q2 * q2))
        z = torch.atan2(2 * (q0 * q3 - q1 * q2), 1 - 2 * (q1 * q1 + q3 * q3))
    elif order == "xzy":
        x = torch.atan2(2 * (q0 * q1 + q2 * q3), 1 - 2 * (q1 * q1 + q3 * q3))
        y = torch.atan2(2 * (q0 * q2 + q1 * q3), 1 - 2 * (q2 * q2 + q3 * q3))
        z = torch.asin(torch.clamp(2 * (q0 * q3 - q1 * q2), -1 + epsilon, 1 - epsilon))
    elif order == "yxz":
        x = torch.asin(torch.clamp(2 * (q0 * q1 - q2 * q3), -1 + epsilon, 1 - epsilon))
        y = torch.atan2(2 * (q1 * q3 + q0 * q2), 1 - 2 * (q1 * q1 + q2 * q2))
        z = torch.atan2(2 * (q1 * q2 + q0 * q3), 1 - 2 * (q1 * q1 + q3 * q3))
    elif order == "zyx":
        x = torch.atan2(2 * (q0 * q1 + q2 * q3), 1 - 2 * (q1 * q1 + q2 * q2))
        y = torch.asin(torch.clamp(2 * (q0 * q2 - q1 * q3), -1 + epsilon, 1 - epsilon))
        z = torch.atan2(2 * (q0 * q3 + q1 * q2), 1 - 2 * (q2 * q2 + q3 * q3))
    else:
        raise

    if deg:
        return torch.stack((x, y, z), dim=1).view(original_shape) * 180 / np.pi
    else:
        return torch.stack((x, y, z), dim=1).view(original_shape)


# Numpy-backed implementations


def qmul_np(q, r):
    q = torch.from_numpy(q).contiguous().float()
    r = torch.from_numpy(r).contiguous().float()
    return qmul(q, r).numpy()


def qrot_np(q, v):
    q = torch.from_numpy(q).contiguous().float()
    v = torch.from_numpy(v).contiguous().float()
    return qrot(q, v).numpy()


def qeuler_np(q, order, epsilon=0, use_gpu=False):
    if use_gpu:
        q = torch.from_numpy(q).cuda().float()
        return qeuler(q, order, epsilon).cpu().numpy()
    else:
        q = torch.from_numpy(q).contiguous().float()
        return qeuler(q, order, epsilon).numpy()


def qfix(q):
    """
    Enforce quaternion continuity across the time dimension by selecting
    the representation (q or -q) with minimal distance (or, equivalently, maximal dot product)
    between two consecutive frames.

    Expects a tensor of shape (L, J, 4), where L is the sequence length and J is the number of joints.
    Returns a tensor of the same shape.
    """
    assert len(q.shape) == 3
    assert q.shape[-1] == 4

    result = q.copy()
    dot_products = np.sum(q[1:] * q[:-1], axis=2)
    mask = dot_products < 0
    mask = (np.cumsum(mask, axis=0) % 2).astype(bool)
    result[1:][mask] *= -1
    return result


def euler2quat(e, order, deg=True):
    """
    Convert Euler angles to quaternions.
    """
    assert e.shape[-1] == 3

    original_shape = list(e.shape)
    original_shape[-1] = 4

    e = e.view(-1, 3)

    ## if euler angles in degrees
    if deg:
        e = e * np.pi / 180.0

    x = e[:, 0]
    y = e[:, 1]
    z = e[:, 2]

    rx = torch.stack(
        (torch.cos(x / 2), torch.sin(x / 2), torch.zeros_like(x), torch.zeros_like(x)),
        dim=1,
    )
    ry = torch.stack(
        (torch.cos(y / 2), torch.zeros_like(y), torch.sin(y / 2), torch.zeros_like(y)),
        dim=1,
    )
    rz = torch.stack(
        (torch.cos(z / 2), torch.zeros_like(z), torch.zeros_like(z), torch.sin(z / 2)),
        dim=1,
    )

    result = None
    for coord in order:
        if coord == "x":
            r = rx
        elif coord == "y":
            r = ry
        elif coord == "z":
            r = rz
        else:
            raise
        if result is None:
            result = r
        else:
            result = qmul(result, r)

    # Reverse antipodal representation to have a non-negative "w"
    if order in ["xyz", "yzx", "zxy"]:
        result *= -1

    return result.view(original_shape)


def expmap_to_quaternion(e):
    """
    Convert axis-angle rotations (aka exponential maps) to quaternions.
    Stable formula from "Practical Parameterization of Rotations Using the Exponential Map".
    Expects a tensor of shape (*, 3), where * denotes any number of dimensions.
    Returns a tensor of shape (*, 4).
    """
    assert e.shape[-1] == 3

    original_shape = list(e.shape)
    original_shape[-1] = 4
    e = e.reshape(-1, 3)

    theta = np.linalg.norm(e, axis=1).reshape(-1, 1)
    w = np.cos(0.5 * theta).reshape(-1, 1)
    xyz = 0.5 * np.sinc(0.5 * theta / np.pi) * e
    return np.concatenate((w, xyz), axis=1).reshape(original_shape)


def euler_to_quaternion(e, order):
    """
    Convert Euler angles to quaternions.
    """
    assert e.shape[-1] == 3

    original_shape = list(e.shape)
    original_shape[-1] = 4

    e = e.reshape(-1, 3)

    x = e[:, 0]
    y = e[:, 1]
    z = e[:, 2]

    rx = np.stack(
        (np.cos(x / 2), np.sin(x / 2), np.zeros_like(x), np.zeros_like(x)), axis=1
    )
    ry = np.stack(
        (np.cos(y / 2), np.zeros_like(y), np.sin(y / 2), np.zeros_like(y)), axis=1
    )
    rz = np.stack(
        (np.cos(z / 2), np.zeros_like(z), np.zeros_like(z), np.sin(z / 2)), axis=1
    )

    result = None
    for coord in order:
        if coord == "x":
            r = rx
        elif coord == "y":
            r = ry
        elif coord == "z":
            r = rz
        else:
            raise
        if result is None:
            result = r
        else:
            result = qmul_np(result, r)

    # Reverse antipodal representation to have a non-negative "w"
    if order in ["xyz", "yzx", "zxy"]:
        result *= -1

    return result.reshape(original_shape)


def quaternion_to_matrix(quaternions):
    """
    Convert rotations given as quaternions to rotation matrices.
    Args:
        quaternions: quaternions with real part first,
            as tensor of shape (..., 4).
    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    r, i, j, k = torch.unbind(quaternions, -1)
    two_s = 2.0 / (quaternions * quaternions).sum(-1)

    o = torch.stack(
        (
            1 - two_s * (j * j + k * k),
            two_s * (i * j - k * r),
            two_s * (i * k + j * r),
            two_s * (i * j + k * r),
            1 - two_s * (i * i + k * k),
            two_s * (j * k - i * r),
            two_s * (i * k - j * r),
            two_s * (j * k + i * r),
            1 - two_s * (i * i + j * j),
        ),
        -1,
    )
    return o.reshape(quaternions.shape[:-1] + (3, 3))


def quaternion_to_matrix_np(quaternions):
    q = torch.from_numpy(quaternions).contiguous().float()
    return quaternion_to_matrix(q).numpy()


def quaternion_to_cont6d_np(quaternions):
    rotation_mat = quaternion_to_matrix_np(quaternions)
    cont_6d = np.concatenate([rotation_mat[..., 0], rotation_mat[..., 1]], axis=-1)
    return cont_6d


def quaternion_to_cont6d(quaternions):
    rotation_mat = quaternion_to_matrix(quaternions)
    cont_6d = torch.cat([rotation_mat[..., 0], rotation_mat[..., 1]], dim=-1)
    return cont_6d


def cont6d_to_matrix(cont6d):
    assert cont6d.shape[-1] == 6, "The last dimension must be 6"
    x_raw = cont6d[..., 0:3]
    y_raw = cont6d[..., 3:6]

    x = x_raw / torch.norm(x_raw, dim=-1, keepdim=True)
    z = torch.cross(x, y_raw, dim=-1)
    z = z / torch.norm(z, dim=-1, keepdim=True)

    y = torch.cross(z, x, dim=-1)

    x = x[..., None]
    y = y[..., None]
    z = z[..., None]

    mat = torch.cat([x, y, z], dim=-1)
    return mat


def cont6d_to_matrix_np(cont6d):
    q = torch.from_numpy(cont6d).contiguous().float()
    return cont6d_to_matrix(q).numpy()


def matrix_to_quaternion(rotation_matrix):
    """
    Convert rotation matrices to quaternions.

    Args:
        rotation_matrix: Rotation matrices (..., 3, 3)

    Returns:
        Quaternions (..., 4) with real part first
    """
    batch_shape = rotation_matrix.shape[:-2]
    rotation_matrix = rotation_matrix.reshape(-1, 3, 3)

    batch_size = rotation_matrix.shape[0]
    q = torch.zeros(
        batch_size, 4, device=rotation_matrix.device, dtype=rotation_matrix.dtype
    )

    trace = (
        rotation_matrix[:, 0, 0] + rotation_matrix[:, 1, 1] + rotation_matrix[:, 2, 2]
    )

    # Case 1: trace > 0
    mask1 = trace > 0
    s1 = torch.sqrt(trace[mask1] + 1.0) * 2
    q[mask1, 0] = 0.25 * s1
    q[mask1, 1] = (rotation_matrix[mask1, 2, 1] - rotation_matrix[mask1, 1, 2]) / s1
    q[mask1, 2] = (rotation_matrix[mask1, 0, 2] - rotation_matrix[mask1, 2, 0]) / s1
    q[mask1, 3] = (rotation_matrix[mask1, 1, 0] - rotation_matrix[mask1, 0, 1]) / s1

    # Case 2: (R00 > R11) and (R00 > R22)
    mask2 = (
        (~mask1)
        & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 1, 1])
        & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 2, 2])
    )
    s2 = (
        torch.sqrt(
            1.0
            + rotation_matrix[mask2, 0, 0]
            - rotation_matrix[mask2, 1, 1]
            - rotation_matrix[mask2, 2, 2]
        )
        * 2
    )
    q[mask2, 0] = (rotation_matrix[mask2, 2, 1] - rotation_matrix[mask2, 1, 2]) / s2
    q[mask2, 1] = 0.25 * s2
    q[mask2, 2] = (rotation_matrix[mask2, 0, 1] + rotation_matrix[mask2, 1, 0]) / s2
    q[mask2, 3] = (rotation_matrix[mask2, 0, 2] + rotation_matrix[mask2, 2, 0]) / s2

    # Case 3: R11 > R22
    mask3 = (~mask1) & (~mask2) & (rotation_matrix[:, 1, 1] > rotation_matrix[:, 2, 2])
    s3 = (
        torch.sqrt(
            1.0
            + rotation_matrix[mask3, 1, 1]
            - rotation_matrix[mask3, 0, 0]
            - rotation_matrix[mask3, 2, 2]
        )
        * 2
    )
    q[mask3, 0] = (rotation_matrix[mask3, 0, 2] - rotation_matrix[mask3, 2, 0]) / s3
    q[mask3, 1] = (rotation_matrix[mask3, 0, 1] + rotation_matrix[mask3, 1, 0]) / s3
    q[mask3, 2] = 0.25 * s3
    q[mask3, 3] = (rotation_matrix[mask3, 1, 2] + rotation_matrix[mask3, 2, 1]) / s3

    # Case 4: else
    mask4 = (~mask1) & (~mask2) & (~mask3)
    s4 = (
        torch.sqrt(
            1.0
            + rotation_matrix[mask4, 2, 2]
            - rotation_matrix[mask4, 0, 0]
            - rotation_matrix[mask4, 1, 1]
        )
        * 2
    )
    q[mask4, 0] = (rotation_matrix[mask4, 1, 0] - rotation_matrix[mask4, 0, 1]) / s4
    q[mask4, 1] = (rotation_matrix[mask4, 0, 2] + rotation_matrix[mask4, 2, 0]) / s4
    q[mask4, 2] = (rotation_matrix[mask4, 1, 2] + rotation_matrix[mask4, 2, 1]) / s4
    q[mask4, 3] = 0.25 * s4

    # Normalize
    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)

    return q.reshape(batch_shape + (4,))


def matrix_to_quaternion_np(rotation_matrix):
    """Numpy version of matrix_to_quaternion."""
    mat = torch.from_numpy(rotation_matrix).contiguous().float()
    return matrix_to_quaternion(mat).numpy()


def cont6d_to_quaternion(cont6d):
    """
    Convert 6D rotation representation to quaternion.

    Args:
        cont6d: 6D rotation (..., 6)

    Returns:
        Quaternions (..., 4) with real part first
    """
    mat = cont6d_to_matrix(cont6d)
    return matrix_to_quaternion(mat)


def cont6d_to_quaternion_np(cont6d):
    """Numpy version of cont6d_to_quaternion."""
    q = torch.from_numpy(cont6d).contiguous().float()
    return cont6d_to_quaternion(q).numpy()


def qpow(q0, t, dtype=torch.float):
    """q0 : tensor of quaternions
    t: tensor of powers
    """
    q0 = qnormalize(q0)
    theta0 = torch.acos(q0[..., 0])

    ## if theta0 is close to zero, add epsilon to avoid NaNs
    mask = (theta0 <= 10e-10) * (theta0 >= -10e-10)
    theta0 = (1 - mask) * theta0 + mask * 10e-10
    v0 = q0[..., 1:] / torch.sin(theta0).view(-1, 1)

    if isinstance(t, torch.Tensor):
        q = torch.zeros(t.shape + q0.shape)
        theta = t.view(-1, 1) * theta0.view(1, -1)
    else:  ## if t is a number
        q = torch.zeros(q0.shape)
        theta = t * theta0

    q[..., 0] = torch.cos(theta)
    q[..., 1:] = v0 * torch.sin(theta).unsqueeze(-1)

    return q.to(dtype)


def qslerp(q0, q1, t):
    """
    q0: starting quaternion
    q1: ending quaternion
    t: array of points along the way

    Returns:
    Tensor of Slerps: t.shape + q0.shape
    """

    q0 = qnormalize(q0)
    q1 = qnormalize(q1)
    q_ = qpow(qmul(q1, qinv(q0)), t)

    return qmul(
        q_,
        q0.contiguous()
        .view(torch.Size([1] * len(t.shape)) + q0.shape)
        .expand(t.shape + q0.shape)
        .contiguous(),
    )


def qbetween(v0, v1):
    """
    find the quaternion used to rotate v0 to v1
    """
    assert v0.shape[-1] == 3, "v0 must be of the shape (*, 3)"
    assert v1.shape[-1] == 3, "v1 must be of the shape (*, 3)"

    v = torch.cross(v0, v1)
    w = torch.sqrt(
        (v0**2).sum(dim=-1, keepdim=True) * (v1**2).sum(dim=-1, keepdim=True)
    ) + (v0 * v1).sum(dim=-1, keepdim=True)
    return qnormalize(torch.cat([w, v], dim=-1))


def qbetween_np(v0, v1):
    """
    find the quaternion used to rotate v0 to v1
    """
    assert v0.shape[-1] == 3, "v0 must be of the shape (*, 3)"
    assert v1.shape[-1] == 3, "v1 must be of the shape (*, 3)"

    v0 = torch.from_numpy(v0).float()
    v1 = torch.from_numpy(v1).float()
    return qbetween(v0, v1).numpy()


def lerp(p0, p1, t):
    if not isinstance(t, torch.Tensor):
        t = torch.Tensor([t])

    new_shape = t.shape + p0.shape
    new_view_t = t.shape + torch.Size([1] * len(p0.shape))
    new_view_p = torch.Size([1] * len(t.shape)) + p0.shape
    p0 = p0.view(new_view_p).expand(new_shape)
    p1 = p1.view(new_view_p).expand(new_shape)
    t = t.view(new_view_t).expand(new_shape)

    return p0 + t * (p1 - p0)
''',
    "utils/motion_utils.py": '''"""
Motion Processing and Feature Conversion Utilities for Human Motion Animation Generation.

271D Feature Format (Pure PyTorch) - Updated per normalization plan:
- Root height Y, Root velocity X, Root velocity Z (3D) - velocity form for X,Z
- 22 RIC positions (66D) - local positions relative to root, from actual data
- 22 6D rotations (132D) - auxiliary features from IK
- 22 local velocities (66D) - causal velocities (current - previous)
- 4D foot contacts - binary contact flags

Total: 271D per frame

Note: Root X,Z are stored as velocities for autoregressive stability.

API:
- preprocess_sequence(): Dataset preprocessing (ground truth joints)
- features_to_positions(): Reconstruction (features -> positions)
- extract_features_from_predicted(): Inference (predicted joints)
- IncrementalFeatureExtractor: Frame-by-frame inference
"""

import torch
import numpy as np
from typing import List, Tuple, Dict, Any, Optional
# Skeleton import removed - not needed
from .quaternion import (
    qrot,
    qinv,
    qmul,
    quaternion_to_cont6d,
    cont6d_to_matrix,
    cont6d_to_quaternion,
)


# ============================================================================
# Skeleton Definitions
# ============================================================================

T2M_RAW_OFFSETS = torch.tensor(
    [
        [0, 0, 0],
        [1, 0, 0],
        [-1, 0, 0],
        [0, 1, 0],
        [0, -1, 0],
        [0, -1, 0],
        [0, 1, 0],
        [0, -1, 0],
        [0, -1, 0],
        [0, 1, 0],
        [0, 0, 1],
        [0, 0, 1],
        [0, 1, 0],
        [1, 0, 0],
        [-1, 0, 0],
        [0, 0, 1],
        [0, -1, 0],
        [0, -1, 0],
        [0, -1, 0],
        [0, -1, 0],
        [0, -1, 0],
        [0, -1, 0],
    ],
    dtype=torch.float32,
)

T2M_KINEMATIC_CHAIN = [
    [0, 2, 5, 8, 11],  # Left leg
    [0, 1, 4, 7, 10],  # Right leg
    [0, 3, 6, 9, 12, 15],  # Spine
    [9, 14, 17, 19, 21],  # Right arm
    [9, 13, 16, 18, 20],  # Left arm
]


# ============================================================================
# Dataset Configuration
# ============================================================================

DATASET_CONFIGS = {
    "t2m": {
        "name": "HumanML3D",
        "num_joints": 22,
        "feature_dim": 271,
        "raw_offsets": T2M_RAW_OFFSETS,
        "kinematic_chain": T2M_KINEMATIC_CHAIN,
        "face_joint_indx": [2, 1, 17, 16],  # [r_hip, l_hip, sdr_r, sdr_l]
        "fid_r": [8, 11],  # Right foot indices
        "fid_l": [7, 10],  # Left foot indices
    },
}


def get_dataset_config(dataset_type: str = "t2m") -> Dict[str, Any]:
    """Get configuration for a dataset type."""
    if dataset_type not in DATASET_CONFIGS:
        raise ValueError(
            f"Unknown dataset_type: {dataset_type}. Available: {list(DATASET_CONFIGS.keys())}"
        )
    return DATASET_CONFIGS[dataset_type]


# ============================================================================
# Feature Layout Constants
# ============================================================================

FEATURE_SLICES = {
    "root_features": slice(0, 3),  # 3D: root height Y, velocity X, velocity Z
    "ric_positions": slice(3, 69),  # 66D (22 * 3)
    "rotations_6d": slice(69, 201),  # 132D (22 * 6)
    "local_velocities": slice(201, 267),  # 66D (22 * 3)
    "foot_contacts": slice(267, 271),  # 4D
}


# ============================================================================
# Internal Helper Functions
# ============================================================================


def _compute_ik(
    positions: torch.Tensor,
    raw_offsets: torch.Tensor,
    kinematic_chain: List[List[int]],
    face_joint_indx: List[int],
) -> torch.Tensor:
    """
    Compute inverse kinematics using pure PyTorch.

    Args:
        positions: Joint positions (..., 22, 3)
        raw_offsets: Skeleton bone offsets (22, 3)
        kinematic_chain: Skeleton kinematic chain
        face_joint_indx: Joint indices for facing direction [r_hip, l_hip, sdr_r, sdr_l]

    Returns:
        Quaternions (..., 22, 4)
    """
    batch_shape = positions.shape[:-2]
    device = positions.device
    dtype = positions.dtype

    # Flatten batch dimensions
    positions_flat = positions.reshape(-1, 22, 3)
    B = positions_flat.shape[0]

    # Get forward direction
    l_hip, r_hip, sdr_r, sdr_l = face_joint_indx

    across1 = positions_flat[:, r_hip] - positions_flat[:, l_hip]  # (B, 3)
    across2 = positions_flat[:, sdr_r] - positions_flat[:, sdr_l]  # (B, 3)
    across = across1 + across2
    across = across / (torch.norm(across, dim=-1, keepdim=True) + 1e-10)

    # Forward direction (cross with Y-up)
    forward = torch.cross(
        torch.tensor([[0, 1, 0]], device=device, dtype=dtype).expand(B, -1),
        across,
        dim=-1,
    )
    forward = forward / (torch.norm(forward, dim=-1, keepdim=True) + 1e-10)

    # Target forward direction (Z-axis)
    target = torch.tensor([[0, 0, 1]], device=device, dtype=dtype).expand(B, -1)

    # Root rotation (from forward to target)
    root_quat = _qbetween(forward, target)

    # Initialize quaternions
    quaternions = torch.zeros(B, 22, 4, device=device, dtype=dtype)
    quaternions[:, 0] = root_quat

    # IK for each chain
    offsets = raw_offsets.unsqueeze(0).expand(B, -1, -1)  # (B, 22, 3)

    for chain in kinematic_chain:
        R = root_quat
        for i in range(len(chain) - 1):
            parent_idx = chain[i]
            child_idx = chain[i + 1]

            # Get bone direction in T-pose
            u = offsets[:, child_idx]  # (B, 3)

            # Get bone direction in current pose
            v = positions_flat[:, child_idx] - positions_flat[:, parent_idx]
            v = v / (torch.norm(v, dim=-1, keepdim=True) + 1e-10)

            # Rotation from u to v
            rot_u_v = _qbetween(u, v)

            # Local rotation
            R_loc = qmul(qinv(R), rot_u_v)

            quaternions[:, child_idx] = R_loc
            R = qmul(R, R_loc)

    return quaternions.reshape(batch_shape + (22, 4))


def _qbetween(v0: torch.Tensor, v1: torch.Tensor) -> torch.Tensor:
    """
    Compute quaternion that rotates v0 to v1.

    Args:
        v0: Source vectors (..., 3)
        v1: Target vectors (..., 3)

    Returns:
        Quaternions (..., 4)
    """
    # Normalize
    v0 = v0 / (torch.norm(v0, dim=-1, keepdim=True) + 1e-10)
    v1 = v1 / (torch.norm(v1, dim=-1, keepdim=True) + 1e-10)

    # Compute rotation
    dot = (v0 * v1).sum(dim=-1, keepdim=True)

    # Handle parallel vectors
    cross = torch.cross(v0, v1, dim=-1)
    w = 1.0 + dot

    q = torch.cat([w, cross], dim=-1)
    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)

    return q


def _forward_kinematics(
    rotations_6d: torch.Tensor,
    root_pos: torch.Tensor,
    offsets: torch.Tensor,
    kinematic_chain: List[List[int]],
) -> torch.Tensor:
    """
    Forward kinematics with 6D rotations (Pure PyTorch).

    Args:
        rotations_6d: 6D rotations (..., 22, 6)
        root_pos: Root position (..., 3)
        offsets: Bone offsets (22, 3)
        kinematic_chain: Skeleton kinematic chain

    Returns:
        Joint positions (..., 22, 3)
    """
    batch_shape = rotations_6d.shape[:-2]
    device = rotations_6d.device
    dtype = rotations_6d.dtype

    # Flatten batch
    rotations_flat = rotations_6d.reshape(-1, 22, 6)
    root_pos_flat = root_pos.reshape(-1, 3)
    B = rotations_flat.shape[0]

    # Initialize positions
    positions = torch.zeros(B, 22, 3, device=device, dtype=dtype)
    positions[:, 0] = root_pos_flat

    # Expand offsets
    offsets_expanded = offsets.unsqueeze(0).expand(B, -1, -1)

    # FK for each chain
    for chain in kinematic_chain:
        # Start with root rotation matrix
        matR = cont6d_to_matrix(rotations_flat[:, 0])  # (B, 3, 3)

        for i in range(1, len(chain)):
            child_idx = chain[i]
            parent_idx = chain[i - 1]

            # Accumulate rotation
            child_rot = cont6d_to_matrix(rotations_flat[:, child_idx])
            matR = torch.bmm(matR, child_rot)

            # Compute position
            offset_vec = offsets_expanded[:, child_idx].unsqueeze(-1)  # (B, 3, 1)
            positions[:, child_idx] = (
                torch.bmm(matR, offset_vec).squeeze(-1) + positions[:, parent_idx]
            )

    return positions.reshape(batch_shape + (22, 3))


# ============================================================================
# Canonical API Functions
# ============================================================================


def preprocess_sequence(
    positions: torch.Tensor,
    dataset_type: str = "t2m",
    feet_thre: float = 0.002,
) -> torch.Tensor:
    """
    Preprocess a sequence of joint positions to 271D features.

    Used for dataset preprocessing with ground truth joints.
    Uses direct RIC transform for perfect round-trip reconstruction.

    Feature Layout (updated per normalization plan):
        [0:3]   Root height Y, Root velocity X, Root velocity Z
        [3:69]  22 RIC positions (22 * 3)
        [69:201] 22 6D rotations (22 * 6)
        [201:267] 22 local velocities (22 * 3)
        [267:271] Foot contacts (4D)

    Note: Root X,Z are stored as velocities for autoregressive stability.

    Args:
        positions: Joint positions (N, 22, 3) or (B, N, 22, 3)
        dataset_type: Dataset type ("t2m" for HumanML3D)
        feet_thre: Foot contact detection threshold

    Returns:
        Feature vectors (N, 271) or (B, N, 271)
    """
    config = get_dataset_config(dataset_type)
    raw_offsets = config["raw_offsets"]
    kinematic_chain = config["kinematic_chain"]
    face_joint_indx = config["face_joint_indx"]
    fid_r = config["fid_r"]
    fid_l = config["fid_l"]

    device = positions.device
    dtype = positions.dtype

    # Handle batch dimension
    if positions.ndim == 4:
        B, N, J, _ = positions.shape
        features_batch = []

        for b in range(B):
            pos_b = positions[b]
            feat_b = preprocess_sequence(pos_b, dataset_type, feet_thre)
            features_batch.append(feat_b)

        return torch.stack(features_batch, dim=0)

    # Single sequence mode: (N, 22, 3)
    N = positions.shape[0]

    # 1. Root features: height Y (absolute), velocity X, velocity Z
    # Per normalization plan: convert root XZ to velocity form
    root_features = torch.zeros(N, 3, device=device, dtype=dtype)
    root_features[:, 0] = positions[:, 0, 1]  # root height Y (keep absolute) at index 0
    if N > 1:
        root_features[1:, 1] = (
            positions[1:, 0, 0] - positions[:-1, 0, 0]
        )  # vx at index 1
        root_features[1:, 2] = (
            positions[1:, 0, 2] - positions[:-1, 0, 2]
        )  # vz at index 2

    # 2. IK for all frames
    quaternions = _compute_ik(
        positions, raw_offsets, kinematic_chain, face_joint_indx
    )  # (N, 22, 4)

    # 3. Root rotation
    root_quat = quaternions[:, 0].clone()  # (N, 4)

    # 4. RIC positions (center all 3 dimensions on root)
    ric = positions - positions[:, 0:1, :]  # (N, 22, 3)
    ric = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)  # (N, 22, 3)

    # 5. 6D rotations
    rotations_6d = quaternion_to_cont6d(quaternions)  # (N, 22, 6)

    # 6. Causal velocities (backward differences)
    local_vel = torch.zeros(N, 22, 3, device=device, dtype=dtype)
    if N > 1:
        local_vel[1:] = qrot(
            root_quat[1:].unsqueeze(1).expand(-1, 22, -1),
            positions[1:] - positions[:-1],
        )

    # 7. Foot contacts
    feet_l = torch.zeros(N, 2, device=device, dtype=dtype)
    feet_r = torch.zeros(N, 2, device=device, dtype=dtype)

    if N > 1:
        vel_l = positions[1:, fid_l] - positions[:-1, fid_l]
        vel_r = positions[1:, fid_r] - positions[:-1, fid_r]
        feet_l[1:] = (torch.sum(vel_l**2, dim=-1) < feet_thre).float()
        feet_r[1:] = (torch.sum(vel_r**2, dim=-1) < feet_thre).float()

    # Concatenate all features
    features = torch.cat(
        [
            root_features,  # [0:3] Root height Y, velocity X, velocity Z
            ric.reshape(N, -1),
            rotations_6d.reshape(N, -1),
            local_vel.reshape(N, -1),
            feet_l,
            feet_r,
        ],
        dim=-1,
    )

    return features


def features_to_positions(
    features: torch.Tensor,
    dataset_type: str = "t2m",
) -> torch.Tensor:
    """
    Reconstruct global joint positions from 271D features.

    Uses direct RIC transform for perfect reconstruction.
    This is the canonical reconstruction function.

    Note: Features now use velocity form for root X,Z. Reconstruction
    requires cumulative sum to recover absolute positions.

    Args:
        features: Feature vectors (..., 271)
        dataset_type: Dataset type ("t2m" for HumanML3D)

    Returns:
        Global joint positions (..., 22, 3)
    """
    # Extract components
    root_features = features[..., 0:3]  # root height Y, velocity X, velocity Z
    ric = features[..., 3:69].reshape(features.shape[:-1] + (22, 3))
    rotations_6d = features[..., 69:201].reshape(features.shape[:-1] + (22, 6))

    # Get root quaternion from 6D rotation
    root_quat = cont6d_to_quaternion(rotations_6d[..., 0, :])  # (..., 4)

    # Reconstruct root position from velocity form
    # root_features[..., 0] = root height Y (absolute)
    # root_features[..., 1] = root velocity X
    # root_features[..., 2] = root velocity Z
    root_height_y = root_features[..., 0:1]  # (..., 1)
    root_vel_x = root_features[..., 1:2]  # (..., 1)
    root_vel_z = root_features[..., 2:3]  # (..., 1)

    # Cumulative sum to recover absolute X and Z positions
    root_pos_x = torch.cumsum(root_vel_x, dim=-1)
    root_pos_z = torch.cumsum(root_vel_z, dim=-1)
    global_root_pos = torch.cat([root_pos_x, root_height_y, root_pos_z], dim=-1)

    # Direct transform: RIC -> global
    # global = root_pos + rotate_inverse(RIC, root_rot)
    root_quat_expanded = root_quat.unsqueeze(-2).expand(root_quat.shape[:-1] + (22, -1))
    positions = global_root_pos.unsqueeze(-2) + qrot(qinv(root_quat_expanded), ric)

    return positions


# ============================================================================
# Incremental Feature Extractor for Autoregressive Generation
# ============================================================================


class IncrementalFeatureExtractor:
    """
    Stateful incremental feature extractor for frame-by-frame generation.

    Used during inference/motion generation when processing one frame at a time.
    Uses FK-based extraction for predicted joints to maintain kinematic consistency.

    Feature Layout (271D) - Updated per normalization plan:
        [0:3]   Root height Y, Root velocity X, Root velocity Z
        [3:69]  22 RIC positions
        [69:201] 22 6D rotations
        [201:267] 22 local velocities
        [267:271] Foot contacts

    Note: Root X,Z are stored as velocities for autoregressive stability.
    """

    def __init__(
        self,
        dataset_type: str = "t2m",
        feet_thre: float = 0.002,
        device: torch.device = torch.device("cpu"),
        dtype: torch.dtype = torch.float32,
    ):
        """
        Initialize the incremental feature extractor.

        Args:
            dataset_type: Dataset type ("t2m" for HumanML3D)
            feet_thre: Foot contact threshold
            device: Torch device
            dtype: Torch dtype
        """
        config = get_dataset_config(dataset_type)
        self.raw_offsets = config["raw_offsets"].to(device).to(dtype)
        self.kinematic_chain = config["kinematic_chain"]
        self.face_joint_indx = config["face_joint_indx"]
        self.fid_r = config["fid_r"]
        self.fid_l = config["fid_l"]
        self.feet_thre = feet_thre
        self.device = device
        self.dtype = dtype

        # State for incremental extraction
        self.prev_positions: Optional[torch.Tensor] = None
        self.is_initialized = False

    def initialize(self, initial_positions: torch.Tensor) -> torch.Tensor:
        """
        Initialize extractor with initial frame positions.

        Args:
            initial_positions: (B, 22, 3) initial joint positions

        Returns:
            Zero features for first frame (B, 271)
        """
        initial_positions = initial_positions.to(self.device).to(self.dtype)
        B = initial_positions.shape[0]

        # Store state
        self.prev_positions = initial_positions.clone()

        # Compute IK for initial frame
        quaternions = _compute_ik(
            initial_positions,
            self.raw_offsets,
            self.kinematic_chain,
            self.face_joint_indx,
        )

        # Compute FK for RIC consistency
        rotations_6d = quaternion_to_cont6d(quaternions)
        root_pos = initial_positions[:, 0]
        fk_positions = _forward_kinematics(
            rotations_6d, root_pos, self.raw_offsets, self.kinematic_chain
        )

        # Store FK positions for next frame's velocity computation
        self.prev_fk_positions = fk_positions

        self.is_initialized = True

        # Return zero features for first frame
        return torch.zeros(B, 271, device=self.device, dtype=self.dtype)

    def process_frame(self, positions: torch.Tensor) -> torch.Tensor:
        """
        Process a single frame and extract 271D features.

        Args:
            positions: (B, 22, 3) joint positions for current frame

        Returns:
            features: (B, 271) feature vectors
        """
        positions = positions.to(self.device).to(self.dtype)

        if not self.is_initialized:
            return self.initialize(positions)

        B = positions.shape[0]

        # === 1. Root features: height Y (absolute), velocity X, velocity Z ===
        root_height_y = positions[:, 0, 1:2]  # (B, 1)
        root_vel_x = positions[:, 0, 0:1] - self.prev_positions[:, 0, 0:1]  # (B, 1)
        root_vel_z = positions[:, 0, 2:3] - self.prev_positions[:, 0, 2:3]  # (B, 1)
        root_features = torch.cat(
            [root_height_y, root_vel_x, root_vel_z], dim=-1
        )  # (B, 3)

        # === 2. IK for rotations ===
        quaternions = _compute_ik(
            positions, self.raw_offsets, self.kinematic_chain, self.face_joint_indx
        )

        # === 3. Root rotation ===
        root_quat = quaternions[:, 0]  # (B, 4)

        # === 4. RIC from FK positions ===
        rotations_6d = quaternion_to_cont6d(quaternions)

        # Get root position for FK (need absolute position)
        global_root_pos = positions[:, 0]  # (B, 3)

        # Compute FK for kinematic consistency
        fk_positions = _forward_kinematics(
            rotations_6d, global_root_pos, self.raw_offsets, self.kinematic_chain
        )

        # Compute RIC from FK positions
        ric = fk_positions - fk_positions[:, 0:1]
        ric = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)

        # === 5. Causal velocities ===
        local_vel = qrot(
            root_quat.unsqueeze(1).expand(-1, 22, -1), positions - self.prev_positions
        )

        # === 6. Foot contacts ===
        foot_vel = positions - self.prev_positions
        feet_l = (
            torch.sum(foot_vel[:, self.fid_l] ** 2, dim=-1) < self.feet_thre
        ).float()
        feet_r = (
            torch.sum(foot_vel[:, self.fid_r] ** 2, dim=-1) < self.feet_thre
        ).float()

        # === Update state ===
        self.prev_positions = positions.clone()

        # === Concatenate features ===
        features = torch.cat(
            [
                root_features,  # [0:3] Root height Y, velocity X, velocity Z
                ric.reshape(B, -1),
                rotations_6d.reshape(B, -1),
                local_vel.reshape(B, -1),
                feet_l,
                feet_r,
            ],
            dim=-1,
        )

        return features

    def reset(self):
        """Reset the extractor state."""
        self.prev_positions = None
        self.prev_fk_positions = None
        self.is_initialized = False


# ============================================================================
# Utility Functions
# ============================================================================


def get_feature_subset(
    features: torch.Tensor,
    subset_names: List[str],
) -> torch.Tensor:
    """
    Extract a subset of features by name.

    Args:
        features: Feature vectors (..., 271)
        subset_names: List of feature names to extract

    Returns:
        Concatenated subset of features
    """
    subsets = []
    for name in subset_names:
        if name not in FEATURE_SLICES:
            raise ValueError(
                f"Unknown feature: {name}. Available: {list(FEATURE_SLICES.keys())}"
            )
        subsets.append(features[..., FEATURE_SLICES[name]])

    return torch.cat(subsets, dim=-1)
''',
}

for filepath, content in FILES.items():
    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Created {filepath}")

# Add to Python path
sys.path.insert(0, str(Path.cwd()))
print("Setup Complete!")

In [ ]:
# --- FEATURE EXTRACTION (Using src/utils imports) ---
import torch
import numpy as np
from utils.motion_utils import (
    T2M_RAW_OFFSETS,
    T2M_KINEMATIC_CHAIN,
    DATASET_CONFIGS,
    get_dataset_config,
    preprocess_sequence,  # 271D format - matches notebook dimension
)
from utils.quaternion import (
    qrot,
    qinv,
    qmul,
    quaternion_to_cont6d,
    quaternion_to_cont6d_np,
)


# Wrapper for 271D feature extraction (matching training pipeline)
def extract_271d_features(
    positions: np.ndarray, feet_thre: float = 0.002
) -> np.ndarray:
    """
    Extract 271D features from joint positions.
    Uses the canonical implementation from motion_utils.py.

    Feature Layout:
        [0:3]   Root height Y, velocity X, velocity Z
        [3:69]  22 RIC positions (22 * 3)
        [69:201] 22 6D rotations (22 * 6)
        [201:267] 22 local velocities (22 * 3)
        [267:271] Foot contacts (4D)
    """
    positions_torch = torch.from_numpy(positions).float()
    features_torch = preprocess_sequence(positions_torch, feet_thre=feet_thre)
    return features_torch.numpy()


print("Feature extraction functions loaded from src/utils/")
print(
    f"Using correct quaternion rotation: qrot(root_quat, ric) - matches MoMask convention"
)

In [ ]:
from concurrent.futures import ThreadPoolExecutor


# --- Create target directories ---
subdirs = ["new_joints", "texts", "new_joint_vecs"]

for sd in subdirs:
    os.makedirs(os.path.join(TARGET_DIR, sd), exist_ok=True)


def copy_files_for_id(i):
    # Copy joint positions
    src_joints = os.path.join(SOURCE_DIR, "new_joints", f"{i}.npy")
    if os.path.exists(src_joints):
        # Copy joint positions
        shutil.copy(src_joints, os.path.join(TARGET_DIR, "new_joints", f"{i}.npy"))

        # Extract 271D features and save to new_joint_vecs
        positions = np.load(src_joints)
        features = extract_271d_features(positions)
        np.save(os.path.join(TARGET_DIR, "new_joint_vecs", f"{i}.npy"), features)

    # Copy text files
    src_text = os.path.join(SOURCE_DIR, "texts", f"{i}.txt")
    if os.path.exists(src_text):
        shutil.copy(src_text, os.path.join(TARGET_DIR, "texts", f"{i}.txt"))


print("\nCopying files and extracting 271D features for subset IDs in parallel...")

with ThreadPoolExecutor() as executor:
    list(tqdm(executor.map(copy_files_for_id, subset_ids), total=len(subset_ids)))


print("\nFile copying and feature extraction complete.")

In [ ]:
# --- Compute Mean and Std from extracted 271D features ---

print("Computing Mean and Std for 271D features...")

# Collect all features
all_features = []
for i in tqdm(subset_ids, desc="Loading features"):
    feat_path = os.path.join(TARGET_DIR, "new_joint_vecs", f"{i}.npy")
    if os.path.exists(feat_path):
        all_features.append(np.load(feat_path))

# Stack and compute statistics
all_features_stacked = np.concatenate(all_features, axis=0)  # (Total_frames, 271)
mean = all_features_stacked.mean(axis=0)
std = all_features_stacked.std(axis=0)

# Per normalization plan: Foot contacts (dims 267:271) should NOT be normalized
# Set mean=0, std=1 for contact dimensions to keep them unchanged
mean[267:271] = 0.0
std[267:271] = 1.0

# Add epsilon to std for numerical stability (per normalization plan)
std = std + 1e-6

# Save
np.save(os.path.join(TARGET_DIR, "Mean.npy"), mean)
np.save(os.path.join(TARGET_DIR, "Std.npy"), std)

print(f"Mean shape: {mean.shape}, Std shape: {std.shape}")
print(f"Total frames processed: {all_features_stacked.shape[0]}")

In [ ]:
# --- ROUND-TRIP VALIDATION ---
print("\n--- Round-trip validation ---")

# Test on first 10 samples
validation_samples = subset_ids[:10]
max_error = 0

for i in tqdm(validation_samples, desc="Validating features"):
    # Load original joint positions
    src_joints = os.path.join(SOURCE_DIR, "new_joints", f"{i}.npy")
    if not os.path.exists(src_joints):
        continue

    original_positions = np.load(src_joints)

    # Extract features
    features = extract_271d_features(original_positions)

    # Verify features are valid 271D vectors
    assert features.shape == (
        original_positions.shape[0],
        271,
    ), f"Feature shape mismatch: {features.shape}"

    # Check for NaN or Inf
    assert not np.any(np.isnan(features)), f"NaN detected in features for {i}"
    assert not np.any(np.isinf(features)), f"Inf detected in features for {i}"

    # Check value ranges are reasonable
    assert np.all(np.abs(features) < 1000), f"Unreasonable values in features for {i}"

print("Round-trip validation passed!")
print(f"  - All features have correct shape (N, 271)")
print(f"  - No NaN or Inf values detected")
print(f"  - All values within reasonable range")

In [ ]:
# --- Update metadata files ---

metadata_files = ["all.txt", "train.txt", "val.txt", "test.txt"]

all_ids = subset_ids.copy()

random.seed(SEED)

random.shuffle(all_ids)


train_idx = int(0.8 * len(all_ids))

val_idx = int(0.9 * len(all_ids))


train_list = sorted(all_ids[:train_idx])

val_list = sorted(all_ids[train_idx:val_idx])

test_list = sorted(all_ids[val_idx:])


for filename in metadata_files:

    with open(os.path.join(TARGET_DIR, filename), "w") as f:

        if filename == "all.txt":

            f.writelines(f"{i}\n" for i in subset_ids)

        elif filename == "train.txt":

            f.writelines(f"{i}\n" for i in train_list)

        elif filename == "val.txt":

            f.writelines(f"{i}\n" for i in val_list)

        elif filename == "test.txt":

            f.writelines(f"{i}\n" for i in test_list)


print("Metadata files updated.")
print(f"  - train.txt: {len(train_list)} samples")
print(f"  - val.txt: {len(val_list)} samples")
print(f"  - test.txt: {len(test_list)} samples")

In [ ]:
# --- Verify 271D features in new_joint_vecs ---
print("Verifying 271D features...")

# Load a sample feature file
sample_feat = np.load(f"{TARGET_DIR}new_joint_vecs/{subset_ids[0]}.npy")
print(f"Feature shape: {sample_feat.shape}")
print(f"Feature dtype: {sample_feat.dtype}")

# Verify feature layout
print(f"\nFeature layout verification:")
print(f"  [0:3]   Root height Y, velocity X, velocity Z: {sample_feat[0, 0:3]}")
print(f"  [3:69]  RIC positions shape: {sample_feat[0, 3:69].shape}")
print(f"  [69:201] 6D rotations shape: {sample_feat[0, 69:201].shape}")
print(f"  [201:267] Local velocities shape: {sample_feat[0, 201:267].shape}")
print(f"  [267:271] Foot contacts: {sample_feat[0, 267:271]}")

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any
import json
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
from IPython.display import HTML

# Missing HumanML3D Kinematic Chain (Standard SMPL/HumanML3D)
KINEMATIC_CHAIN = [
    [0, 2, 5, 8, 11],  # Right leg
    [0, 1, 4, 7, 10],  # Left leg
    [0, 3, 6, 9, 12, 15],  # Spine
    [9, 14, 17, 19, 21],  # Right arm
    [9, 13, 16, 18, 20],  # Left arm
]


def plot_3d_motion(
    motion: np.ndarray,
    fps: int = 20,
    radius: float = 1.0,
    title: str = "Motion Visualization",
    follow_root: bool = False,
) -> FuncAnimation:
    """
    Create a 3D animation of motion joint positions.
    """
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection="3d")
    ax.view_init(elev=20, azim=45)

    colors = ["#2980b9", "#c0392b", "#27ae60", "#f39c12", "#8e44ad"]
    lines = [ax.plot([], [], [], color=c, marker="o", ms=2, lw=2)[0] for c in colors]

    ax.set_xlabel("X (Side)")
    ax.set_ylabel("Z (Forward)")
    ax.set_zlabel("Y (Height)")
    ax.set_title(title)

    # Pre-calculate global bounds for static centering if not following root
    pos_min = motion.min(axis=(0, 1))
    pos_max = motion.max(axis=(0, 1))

    def update(frame):
        root = motion[frame, 0, :]

        if follow_root:
            ax.set_xlim3d([root[0] - radius, root[0] + radius])
            ax.set_ylim3d([root[2] - radius, root[2] + radius])
            ax.set_zlim3d([pos_min[1], pos_max[1] + radius * 0.5])
        else:
            ax.set_xlim3d([pos_min[0] - radius, pos_max[0] + radius])
            ax.set_ylim3d([pos_min[2] - radius, pos_max[2] + radius])
            ax.set_zlim3d([pos_min[1], pos_max[1] + radius * 0.5])

        for i, c_indices in enumerate(KINEMATIC_CHAIN):
            joints = motion[frame, c_indices, :]
            # Map Data Y to Plot Z (Vertical)
            lines[i].set_data(joints[:, 0], joints[:, 2])
            lines[i].set_3d_properties(joints[:, 1])
        return lines

    ani = FuncAnimation(
        fig, update, frames=len(motion), interval=1000 / fps, blit=False
    )
    plt.close()
    return ani


def visualize_motion(
    joint_positions: np.ndarray,
    ground_truth: Optional[np.ndarray] = None,
    title: str = "Motion Visualization",
    save_path: Optional[Path] = None,
    fps: int = 20,
    skip_frames: int = 1,
    notebook: bool = True,
) -> Any:
    """
    Visualize motion from joint positions.
    """
    fps = fps / skip_frames
    ani = plot_3d_motion(joint_positions[::skip_frames], fps=fps, title=title)

    if save_path:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        ani.save(str(save_path), writer="ffmpeg", fps=fps)
        print(f"Saved animation to {save_path}")

    if notebook:
        display_html = HTML(ani.to_html5_video())
        return display_html
    return ani


# 1. Load data
file_id = subset_ids[100]
data_path = f"{TARGET_DIR}new_joints/{file_id}.npy"
motion_data = np.load(data_path)
text_path = f"{TARGET_DIR}texts/{file_id}.txt"
with open(text_path, "r") as f:
    text = f.read()

# 2. Visualize
ani = visualize_motion(motion_data, title=f"{file_id}.npy", fps=20, skip_frames=2)
print(text)
ani